# Phase 3 — Business Confidence Index (BCI)

BCI = How much should we trust this income estimate?

## Components (weighted)
| Component | Weight | What it Measures |
|---|---|---|
| Income Stability | 30% | Regularity of inflows, interval tightness |
| Segment Clarity | 20% | How cleanly customer fits their segment |
| Data Richness | 20% | History depth + transaction density |
| Model Confidence | 20% | Band classifier certainty |
| Behavioral Consistency | 10% | Spend patterns match predicted income |

## BCI Bands → Policy
| Band | Score | Policy | Income Haircut |
|---|---|---|---|
| HIGH | 80–100 | STP | 100% |
| MEDIUM | 60–79 | STP | 80% |
| LOW | 40–59 | Manual Review | 60% |
| VERY LOW | 0–39 | Decline/Refer | 0% |

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.bci import BCIScorer
from src.utils import setup_logging

setup_logging('INFO')

In [ ]:
# Assumes features_df and income_results from previous notebooks
scorer = BCIScorer(config_path='../config/config.yaml')
bci_results = scorer.compute(features_df, income_results, segment_col='segment')
scorer.get_summary(bci_results)

## BCI Distribution by Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bci_results['bci_band'].value_counts().plot(kind='bar', ax=axes[0], title='BCI Band Distribution')

sns.boxplot(data=bci_results, x='segment', y='bci_score', ax=axes[1])
axes[1].set_title('BCI Score by Segment')
axes[1].axhline(60, color='orange', linestyle='--', label='MEDIUM threshold')
axes[1].axhline(80, color='green', linestyle='--', label='HIGH threshold')
axes[1].legend()
plt.tight_layout()
plt.show()

## Component Contribution Analysis

In [ ]:
component_cols = [
    'bci_stability', 'bci_segment_clarity', 'bci_data_richness',
    'bci_model_confidence', 'bci_behavioral_consistency'
]
bci_results.groupby('bci_band')[component_cols].mean().round(3)

## Single Customer Explainability

In [ ]:
# Example: explain BCI for a single customer
sample_id = bci_results.index[0]
scorer.get_component_breakdown(bci_results, sample_id)